# 29. Ders: Kariyer Terzisi'ni Web'e Taşımak (Akıllı Özetleme Arayüzü)

## Adım 1: Teorik Çerçeve ve Günlük Hayat Analojisi

GCA Coding Jam'deki Bulletproof (Kariyer Terzisi) projesinin iki kolu vardı: Biri anahtar kelime sayan ATS sistemi, diğeri ise uzun makaleleri veya özgeçmişleri özetleyen Yapay Zeka botuydu (14. Derste işlemiştik). 

Kullanıcılar kod yazmayı bilmezler; onlara uzun metinlerini yapıştırabilecekleri devasa bir kutu ve çıkan özeti rahatça kopyalayabilecekleri ayrı bir kutu vermeliyiz. Ayrıca kullanıcı bu özetin ne kadar uzun olacağını (Max Length) kendisi seçmek ister. Bu dersimizde Text Summarization (Metin Özetleme) altyapımızı, kullanıcı dostu dinamik kaydırma çubukları ve birden fazla çıktı ekranıyla (Multiple Outputs) donatacağız.

## Adım 2: Teknik Terimler ve Sözlük

- Multiple Outputs (Çoklu Çıktı): Arayüzün sadece tek bir cevap dönmesi yerine, aynı anda birden fazla kutuyu (Örn: Özet Metni kutusu ve Tasarruf Yüzdesi kutusu) doldurmasıdır.
- File Upload (Dosya Yükleme): Gradio'da kullanıcının yazıyı kopyalamak yerine doğrudan `.txt` veya `.pdf` belgesi yüklemesini sağlayan gelişmiş bileşen (Kavramsal).
- Processing Time (İşlem Süresi): Modelin devasa bir metni okuması ve yeni cümleler (Abstractive) yazması uzun süreceği için arayüzde oluşan bekleme süresidir.

## Adım 3: Hugging Face Summarization Modelini Yükleme

Özetleme yapacak devasa NLP (Doğal Dil İşleme) modelimizi içeri alıyoruz.

In [ ]:
import gradio as gr
from transformers import pipeline
import logging

logging.getLogger("transformers").setLevel(logging.ERROR)

# İngilizce özetleme modelimiz (İndirilmesi biraz sürebilir)
ozetleyici_model = pipeline("summarization")

print("Özetleme şefi hazır.")

## Adım 4: Uzun Metin Özetleyen ve Çoklu Veri Döndüren Fonksiyon

Kullanıcı hem metni verecek hem de özetin maksimum kaç kelime (token) olacağını Slider ile belirleyecek. Biz de geriye 2 şey döndüreceğiz: Birincisi özet metnin kendisi, ikincisi ise kullanıcıya ne kadar zaman/kelime tasarrufu sağladığımızın oranı.

In [ ]:
def makaleyi_ozetle(orijinal_metin, max_kelime):
    # Kullanıcı metin girmemişse uyar
    if len(orijinal_metin.strip()) < 50:
        return "Hata: Özetleme için en az 50 karakterlik bir metin girmelisiniz.", "Tasarruf: %0"
        
    # Modeli çalıştır, max_length parametresini kullanıcının Slider'ından (max_kelime) al
    # min_length'i güvenli bir şekilde ayarlayalım ki model hata vermesin
    sonuc = ozetleyici_model(orijinal_metin, max_length=max_kelime, min_length=20, do_sample=False)
    
    ozet_metin = sonuc[0]['summary_text']
    
    # Tasarruf istatistiği hesabı
    uzunluk_orijinal = len(orijinal_metin)
    uzunluk_ozet = len(ozet_metin)
    tasarruf_orani = 100 - ((uzunluk_ozet / uzunluk_orijinal) * 100)
    
    istatistik_metni = f"Okuma yükünden %{tasarruf_orani:.1f} tasarruf edildi."
    
    # DİKKAT: İki farklı şey döndürüyoruz (Tuple olarak döner)
    return ozet_metin, istatistik_metni

## Adım 5: Gradio'da Çoklu Çıktı (Multiple Outputs) Ayarlama

Fonksiyonumuz iki (2) değer döndürdüğü için, arayüzün `outputs` kısmına da iki (2) tane kutu koymak zorundayız. Aksi halde sistem çöker.

In [ ]:
# Girdiler (Liste olarak: Metin Kutusu ve Slider)
girdiler = [
    gr.Textbox(lines=10, placeholder="İngilizce makalenizi buraya yapıştırın...", label="Orijinal Metin"),
    gr.Slider(minimum=30, maximum=150, step=10, value=60, label="Özetin Maksimum Kelime Uzunluğu")
]

# Çıktılar (Liste olarak: Biri özetin yazılacağı kutu, diğeri istatistiğin yazılacağı kutu)
ciktilar = [
    gr.Textbox(lines=4, label="✨ Yapay Zeka Özeti"),
    gr.Textbox(lines=1, label="📊 Tasarruf İstatistiği")
]

arayuz = gr.Interface(
    fn=makaleyi_ozetle,
    inputs=girdiler,
    outputs=ciktilar,
    title="📝 Bulletproof Akıllı Makale Özetleyici",
    description="Okumaya üşendiğiniz uzun İngilizce metinleri veya özgeçmişleri buraya yapıştırın. Yapay zeka sizin için en önemli kısımları çekip kısaltsın."
)

print("Arayüz bileşenleri hazırlandı.")

## Adım 6: Uygulamayı Canlıya Alma (Launch)

Uygulamamızı başlatalım.

In [ ]:
arayuz.launch(share=True)

## Adım 7: Dosya Yükleme (File Component) Üzerine Düşünmek

Gelişmiş Gradio uygulamalarında (gerçek bir Kariyer Terzisi projesi için), kullanıcıdan yazıyı kopyalayıp yapıştırmasını istemeyiz. Onun yerine `gr.File(label="CV'nizi Yükleyin")` bileşeni kullanırız. Kullanıcı PDF yükler, arka planda (Örn: PyPDF2 kütüphanesiyle) PDF'in içindeki yazılar koda çekilir ve model öyle çalıştırılır. Vaktiniz olduğunda Gradio'nun dökümantasyonundan 'File' okuma mantığını araştırabilirsiniz.

## Adım 8: Dene ve Keşfet - Gerçek Verilerle Test Et

Wikipedia'dan (Örn: Artificial Intelligence) uzun bir İngilizce paragraf (en az 10-15 satır) kopyalayın. Linkteki web sitenize girip yapıştırın. Önce slider'ı 150'ye getirip özetletin, sonra 50'ye getirip tekrar özetletin. Modelin zorla kısaltma yaparken hangi hayati cümleleri seçtiğine dikkat edin.

## Adım 9: Sıra Sende Görevi - ATS Motoru Web Arayüzü

Problem: 13. Derste yazdığınız, bir İş İlanı ile bir CV'yi karşılaştırıp % eşleşme skoru (Set Kesişimi) veren anahtar kelime eşleştirici (ATS Motoru) altyapısını, 2 girdili ve tek çıktılı bir Gradio web sitesine dönüştürmeniz isteniyor.

Gereksinimler:
1. İki parametre (`ilan_metni`, `cv_metni`) alan ve geriye sadece bir string (`f"Adayın İlana Uyumu: %{skor}"`) döndüren bir fonksiyon (`ats_arayuz_motoru`) yazın. (13. dersteki NLTK fonksiyonlarını direkt kopyalayıp içine yapıştırabilirsiniz).
2. `gr.Interface` oluşturun.
3. Girdiler (`inputs`): 2 adet `gr.Textbox(lines=5)` olsun. (Biri ilan için, biri CV için).
4. Çıktı (`outputs`): 1 adet `gr.Textbox` olsun.
5. Sistemi `launch(share=True)` ile yayınlayın.

In [ ]:
# GÖREV KODLARINIZI BURAYA YAZINIZ:



Tebrikler! 29. Dersi tamamlayarak, NLP dünyasının en zorlu konularından olan Text Summarization (Özetleme) görevini dinamik değişkenler ve istatistik panelleriyle şık bir ürüne dönüştürdünüz.

Bir sonraki dersimiz olan Büyük Final'de (30. Ders), şimdiye kadar yaptığımız tüm bu yapay zeka projelerini (Kavanoz, Şef, Yazar) TEK BİR web sitesi altında (Sekmeler/Tabs halinde) toplayacak ve kendi sanal Yapay Zeka Şirketinizi kuracaksınız!